In [6]:
import os
import random
import json
import time
import argparse
import platform
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# ==========================================
# 1. Configuration & Utilities
# ==========================================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ==========================================
# 2. Specialists (Mini-ResNet) & Broker
# ==========================================

class ResidualBlock(nn.Module):
    """Standard Residual Block for deeper learning"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class SpectralSpecialist(nn.Module):
    """Deepened Spectral Specialist"""
    def __init__(self, out_channels=64):
        super().__init__()
        # Input 3 -> 32 -> 64 (ResBlock) -> 64 (ResBlock) -> 8x8 Output
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 64, stride=2), # Downsample 16x16
            ResidualBlock(64, out_channels, stride=1),
            nn.AdaptiveAvgPool2d((8, 8)) # Force 8x8 spatial
        )
    def forward(self, x): return self.net(x)

class SpatialSpecialist(nn.Module):
    """Deepened Spatial Specialist"""
    def __init__(self, out_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 64, stride=2), # Downsample 16x16
            ResidualBlock(64, out_channels, stride=1),
            nn.AdaptiveAvgPool2d((8, 8)) # Force 8x8 spatial
        )
    def forward(self, x): return self.net(x)

class KnowledgeBroker(nn.Module):
    def __init__(self, num_specialists, channels, num_classes=10):
        super().__init__()
        self.num_specialists = num_specialists
        self.projections = nn.ModuleList([
            nn.Conv2d(channels, channels, kernel_size=1) for _ in range(num_specialists)
        ])
        
        total_ch = num_specialists * channels
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(total_ch, total_ch // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(total_ch // 4, total_ch, 1),
            nn.Sigmoid()
        )
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )
        self.bottleneck = nn.Sequential(
            nn.Conv2d(total_ch, channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(channels, num_classes)

    def forward(self, features):
        projected_feats = [proj(f) for proj, f in zip(self.projections, features)]
        z_raw = torch.cat(projected_feats, dim=1)

        w_c = self.channel_gate(z_raw)
        z_calibrated = z_raw * w_c
        
        avg_out = torch.mean(z_calibrated, dim=1, keepdim=True)
        max_out, _ = torch.max(z_calibrated, dim=1, keepdim=True)
        w_s = self.spatial_gate(torch.cat([avg_out, max_out], dim=1))
        z_star = z_calibrated * w_s

        f_out = self.bottleneck(z_star)
        pooled = self.pool(f_out).flatten(1)
        logits = self.classifier(pooled)

        return logits, projected_feats

class KFN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.specialists = nn.ModuleList([SpectralSpecialist(), SpatialSpecialist()])
        self.broker = KnowledgeBroker(num_specialists=2, channels=64, num_classes=num_classes)

    def forward(self, x):
        feats = [spec(x) for spec in self.specialists]
        return self.broker(feats)

# ==========================================
# 3. Stabilized CKA Loss
# ==========================================
def cka_loss(feats):
    device = feats[0].device
    n = len(feats)
    if n < 2: return torch.tensor(0.0, device=device)

    def hsic(K, L):
        N = K.shape[0]
        H = torch.eye(N, device=device) - (1.0/N) * torch.ones((N, N), device=device)
        return torch.trace(H @ K @ H @ L) / ((N - 1)**2)

    loss = torch.tensor(0.0, device=device)
    count = 0
    for i in range(n):
        for j in range(i+1, n):
            x = feats[i].flatten(1)
            y = feats[j].flatten(1)
            K = x @ x.t()
            L = y @ y.t()
            hsic_k = hsic(K, K)
            hsic_l = hsic(L, L)
            denom = torch.sqrt(hsic_k * hsic_l + 1e-12)
            metric = hsic(K, L) / (denom + 1e-12)
            loss += metric
            count += 1
    
    return loss / count if count > 0 else torch.tensor(0.0, device=device)

# ==========================================
# 4. Main Execution
# ==========================================
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    parser.add_argument("--beta", type=float, default=0.1, help="CKA strength")
    # UPDATED: Default epochs increased to 50 for meaningful convergence
    parser.add_argument("--epochs", type=int, default=50, help="Epochs")
    
    args, unknown = parser.parse_known_args()

    # CONFIGURATION
    CONFIG = {
        "data_path": r'D:\Research\Knowledge Fusion Network\cifar-10-batches-py',
        "batch_size": 128,
        "epochs": args.epochs,
        "lr": 1e-3, # Increased initial LR for Scheduler
        "seed": args.seed,
        "cka_beta": args.beta,
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S")
    }
    
    os.makedirs("logs", exist_ok=True)
    run_id = f"seed{args.seed}_beta{args.beta}_deep"
    with open(f"logs/config_{run_id}.json", "w") as f:
        json.dump(CONFIG, f, indent=4)
    
    set_deterministic(CONFIG["seed"])
    device = get_device()
    
    # --- UPDATED: Data Augmentation (The Secret Sauce) ---
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])
    
    # Test transform stays clean (No augmentation)
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])
    
    trainset = torchvision.datasets.CIFAR10(root=CONFIG["data_path"], train=True, download=True, transform=train_transform)
    testset = torchvision.datasets.CIFAR10(root=CONFIG["data_path"], train=False, download=True, transform=test_transform)
    
    trainloader = DataLoader(trainset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=4, pin_memory=True)
    testloader = DataLoader(testset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=4, pin_memory=True)
    
    # Model
    model = KFN().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
    
    # --- UPDATED: Scheduler (The Accuracy Booster) ---
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
    criterion = nn.CrossEntropyLoss()
    
    history = {"train_loss": [], "test_acc": [], "cka": [], "epoch_time": [], "lr": []}
    
    print(f"Starting Run: {run_id} (Mini-ResNet + Aug + CosineLR)")
    best_acc = 0.0

    for epoch in range(CONFIG["epochs"]):
        start_time = time.time()
        model.train()
        total_loss, total_cka = 0, 0
        
        for inputs, labels in tqdm(trainloader, desc=f"Ep {epoch+1}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            logits, proj_feats = model(inputs)
            loss_det = criterion(logits, labels)
            loss_reg = cka_loss(proj_feats)
            loss = loss_det + CONFIG["cka_beta"] * loss_reg 
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            total_cka += loss_reg.item()
        
        # Step the scheduler
        scheduler.step()
        
        # Validation
        model.eval()
        correct = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                logits, _ = model(inputs)
                correct += (logits.argmax(1) == labels).sum().item()
        
        test_acc = 100 * correct / len(testset)
        epoch_dur = time.time() - start_time
        current_lr = scheduler.get_last_lr()[0]
        
        history["train_loss"].append(total_loss/len(trainloader))
        history["test_acc"].append(test_acc)
        history["cka"].append(total_cka/len(trainloader))
        history["epoch_time"].append(epoch_dur)
        history["lr"].append(current_lr)
        
        print(f"Ep {epoch+1} | Loss: {history['train_loss'][-1]:.4f} | CKA: {history['cka'][-1]:.4f} | TestAcc: {test_acc:.2f}% | LR: {current_lr:.6f}")
        
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), f"logs/model_{run_id}.pth")

    # Save Results
    with open(f"logs/history_{run_id}.json", "w") as f:
        json.dump(history, f)

    # Save Features
    print("Saving Features...")
    model.load_state_dict(torch.load(f"logs/model_{run_id}.pth"))
    model.eval()
    f1, f2 = [], []
    with torch.no_grad():
        for inputs, _ in testloader:
            inputs = inputs.to(device)
            _, feats = model(inputs)
            f1.append(feats[0].flatten(1).cpu())
            f2.append(feats[1].flatten(1).cpu())
    
    torch.save({
        "spec1": torch.cat(f1),
        "spec2": torch.cat(f2)
    }, f"logs/features_{run_id}.pt")
    
    print(f"Run {run_id} Complete. Best Acc: {best_acc:.2f}%")

Starting Run: seed42_beta0.1_deep (Mini-ResNet + Aug + CosineLR)


Ep 1: 100%|██████████| 391/391 [04:20<00:00,  1.50it/s]


Ep 1 | Loss: 1.6005 | CKA: 0.0885 | TestAcc: 49.67% | LR: 0.000999


Ep 2: 100%|██████████| 391/391 [04:43<00:00,  1.38it/s]


Ep 2 | Loss: 1.2163 | CKA: 0.0891 | TestAcc: 55.47% | LR: 0.000996


...

Ep 48 | Loss: 0.1940 | CKA: 0.0421 | TestAcc: 86.65% | LR: 0.000004


Ep 49: 100%|██████████| 391/391 [05:14<00:00,  1.24it/s]


Ep 49 | Loss: 0.1939 | CKA: 0.0424 | TestAcc: 86.68% | LR: 0.000001


Ep 50: 100%|██████████| 391/391 [04:42<00:00,  1.39it/s]


Ep 50 | Loss: 0.1959 | CKA: 0.0417 | TestAcc: 86.59% | LR: 0.000000
Saving Features...
Run seed42_beta0.1_deep Complete. Best Acc: 86.71%


Part 2: Comparison

In [8]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import json
import os
import time
from tqdm import tqdm

# Standard ResNet18 Baseline (Fair Comparison Mode)
def train_baseline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs("logs", exist_ok=True)
    
    # --- CRITICAL FIX 1: MATCH KFN AUGMENTATION ---
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])
    
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])

    trainset = torchvision.datasets.CIFAR10(root=r'D:\Research\Knowledge Fusion Network\cifar-10-batches-py', train=True, transform=train_transform)
    testset = torchvision.datasets.CIFAR10(root=r'D:\Research\Knowledge Fusion Network\cifar-10-batches-py', train=False, transform=test_transform)
    
    trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=4)
    testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=4)
    
    # Model: Standard ResNet18
    model = torchvision.models.resnet18(num_classes=10).to(device)
    # CIFAR-10 Adaptation (Keep this, it's correct)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    
    # --- CRITICAL FIX 2: MATCH KFN OPTIMIZER SETTINGS ---
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    # --- CRITICAL FIX 3: MATCH KFN EPOCHS & SCHEDULER ---
    epochs = 50
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    history = {"test_acc": [], "train_loss": []}
    
    print("Starting FAIR Baseline ResNet18 Training (Aug + CosineLR)...")
    
    for epoch in range(epochs): 
        model.train()
        total_loss = 0
        for inputs, labels in tqdm(trainloader, desc=f"Ep {epoch+1}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Step Scheduler
        scheduler.step()
            
        model.eval()
        correct = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                correct += (outputs.argmax(1) == labels).sum().item()
        
        acc = 100 * correct / len(testset)
        history["test_acc"].append(acc)
        history["train_loss"].append(total_loss/len(trainloader))
        
        print(f"Baseline Ep {epoch+1} | Loss: {history['train_loss'][-1]:.4f} | Acc: {acc:.2f}%")
        
    with open("logs/baseline_history.json", "w") as f:
        json.dump(history, f)
    
    # Save the baseline model stats too for parameter comparison
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    with open("logs/baseline_stats.txt", "w") as f:
        f.write(f"ResNet18 Parameters: {params}\n")
        
    print(f"Baseline Complete. Final Acc: {acc:.2f}%")

if __name__ == "__main__":
    train_baseline()

Starting FAIR Baseline ResNet18 Training (Aug + CosineLR)...


Ep 1: 100%|██████████| 391/391 [19:32<00:00,  3.00s/it]


Baseline Ep 1 | Loss: 1.3661 | Acc: 58.45%


Ep 2: 100%|██████████| 391/391 [19:37<00:00,  3.01s/it]


Baseline Ep 2 | Loss: 0.8842 | Acc: 70.07%


...

Baseline Ep 48 | Loss: 0.0033 | Acc: 92.77%


Ep 49: 100%|██████████| 391/391 [28:24<00:00,  4.36s/it]


Baseline Ep 49 | Loss: 0.0026 | Acc: 92.73%


Ep 50: 100%|██████████| 391/391 [36:07<00:00,  5.54s/it]


Baseline Ep 50 | Loss: 0.0025 | Acc: 92.76%
Baseline Complete. Final Acc: 92.76%
